# JWST mosaics, PSF matching, and coadds

Level-3 mosaics and PSF-matched coadds via `st123.mosaic` and `st123.align`.
Sky-region parsing uses `st123.mast.parse_s_region`.

CLI equivalent: `python -m st123.scripts.mosaic ...`


In [ ]:
import sys
from pathlib import Path

# Resolve repo root whether cwd is repo, st123/, or st123/notebooks/
_here = Path.cwd().resolve()
ROOT = None
for candidate in [_here, *_here.parents]:
    if (candidate / 'pyproject.toml').is_file() and (candidate / 'st123').is_dir():
        ROOT = candidate
        break
if ROOT is None:
    ROOT = _here
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import shapely
from astropy.io import fits

from st123.mast import parse_s_region
from st123.utils.helpers import input_list
from st123 import jwst_phot


## Input JHAT-aligned frames

Point `workdir` at a reduction that already has `jhat/*jhat.fits` products.


In [ ]:
workdir = Path('jwstred_temp_dolphot')
jhat_dir = workdir / 'jhat'
mosaic_dir = workdir / 'mosaic'
mosaic_dir.mkdir(parents=True, exist_ok=True)

input_images = sorted(str(p) for p in jhat_dir.glob('*jhat.fits'))
print(len(input_images), 'input images under', jhat_dir)

if not input_images:
    raise FileNotFoundError(
        f'No *jhat.fits under {jhat_dir}. Update workdir after running alignment.'
    )

table = input_list(input_images)
table

## Level-3 mosaic

`generate_level3_mosaic` builds a multi-frame Level-3 product.
`create_default_mosaic` restricts to one filter.


In [ ]:
from st123.mosaic import create_default_mosaic
from st123 import generate_level3_mosaic

FILTER = None  # e.g. 'f200w' to mosaic one filter only

if FILTER:
    mosaic_name = create_default_mosaic(
        input_images,
        outdir=str(mosaic_dir),
        filt=FILTER,
    )
else:
    mosaic_name = generate_level3_mosaic(
        input_images,
        outdir=str(workdir / 'out'),
    )

print('Mosaic:', mosaic_name)
mosaic_name

## Sky footprint from S_REGION


In [ ]:
pgons = [
    parse_s_region(fits.open(im)['SCI'].header['S_REGION'])
    for im in table['image']
]
net_field = shapely.unary_union(pgons)
print('union area (deg^2):', net_field.area)
net_field

## Coadd matched mosaics

`coadd` inverse-variance / duration-weights SCI frames and writes SCI/ERR/WHT HDUs.


In [ ]:
from st123.mosaic import coadd

ref_files = sorted(str(p) for p in mosaic_dir.glob('*_i2d.fits'))
if mosaic_name and Path(mosaic_name).is_file() and mosaic_name not in ref_files:
    ref_files.insert(0, mosaic_name)

coadd_path = mosaic_dir / 'coadd_i2d.fits'
if len(ref_files) < 2:
    print(
        'Need at least two mosaic *_i2d.fits under', mosaic_dir,
        'to coadd. Found:', ref_files,
    )
    coadd_path = Path(mosaic_name) if mosaic_name else None
else:
    filt = Path(ref_files[-1]).stem.split('_')[-2].upper()
    coadd(ref_files, filt, filename=str(coadd_path))
    print('Wrote', coadd_path, 'from', len(ref_files), 'inputs; filter=', filt)

coadd_path

## Photometry check on coadd


In [ ]:
if coadd_path is None or not Path(coadd_path).is_file():
    raise FileNotFoundError('No coadd/mosaic FITS available for photometry.')

refcat, photfile = jwst_phot(str(coadd_path))
print('Phot catalog:', photfile, 'n=', len(refcat))

mags = np.asarray(refcat['mag'], dtype=float)
mags = mags[np.isfinite(mags) & (mags > 15) & (mags < 30)]
plt.hist(mags, bins=40)
plt.xlabel('mag')
plt.ylabel('N')
plt.title(Path(coadd_path).name)
plt.show()
refcat[:5]